# Exercise Sheet No. 6

---

> Machine Learning for Natural Sciences, Prof. Pascal Friederich
>
> Deadline: see the course announcement
>
> Tutor: yumeng.zhang@kit.edu
>
> Please ask questions in the forum/discussion board. For grading issues, contact the tutor.
------
**Topic**: This exercise sheet uses neural networks for molecular dynamics simulation.

Please add here your group members' names and student IDs. 

You are encouraged to work in groups of a maximum of 3 people, however **each of you** has to submit a solution.

Names: 

IDs:

# Molecular dynamics simulation

"Molecular dynamics (MD) is a computer simulation method for analyzing the physical movements of atoms and molecules [that are interacting to each other and creating forces among themselves]. (...)  In the most common version, the trajectories (movements) of atoms and molecules are determined by numerically solving Newton's equations of motion for a system of interacting particles, where forces between the particles and their potential energies are often calculated using interatomic potentials or molecular mechanics force fields." [wikipedia](https://en.wikipedia.org/wiki/Molecular_dynamics)

In this exercise, we will perform an MD simulation of a single simple molecule, methanol (CH3OH). The propagation of atomic positions in time is usually treated classically and is given by Newton's equations of motion for an ensemble of particles:

$$ F(X) = - \nabla U(X) = M \dot{V}(t)$$

$$ V(t) = \dot{X}(t) $$

A molecular dynamics simulation therefore requires a potential function $U(X)$, or an equivalent description of the force terms $F(X)$ by which the particles interact. To correctly capture molecular interactions, different methods are used depending on the system:

* (Classical) force fields are empirical energy functions and consist of bonded interactions (associated with chemical bonds, bond angles, and dihedral angles) and non-bonded interactions (associated with van der Waals interaction, Pauli repulsion, and electrostatic interaction).

* Pair potentials between particles calculate the total potential energy from pairwise energy contributions. An example is the non-bonded Lennard-Jones potential.

* Semi-empirical potentials are based on quantum mechanical methods, but use empirical parameterizations to estimate energy contributions, e.g. tight-binding potentials.

* Ab initio molecular dynamics (AIMD) simulations use quantum mechanical methods to compute energies and forces. This is more accurate than classical force fields and can describe chemical processes such as bond breaking or formation, but quantum mechanical methods are much more expensive and therefore limited in system size and time scale.

* QM/MM methods are hybrid methods between quantum mechanical (QM) and molecular or classical mechanics (MM), where only a small, important part is modeled using QM methods.

The goal of this exercise is to model the potential energy with a neural network. Neural networks can be fast in prediction and differentiable, and if trained on quantum mechanical (QM) data, they can approach the accuracy of AIMD/QM methods. To work for arbitrary molecules and intermolecular interactions, neural-network potentials usually need convolutional, graph-based, or atom-centered architectures. Those methods go beyond the scope of this exercise.

By the end of this notebook, you should be able to:

* parse molecular geometries from `.xyz` files,
* explain why raw Cartesian coordinates are problematic inputs for molecular energies,
* implement inverse-distance features that are invariant to translation and rotation,
* connect energy gradients to forces, and
* compare plain Verlet integration with thermostat-controlled dynamics.

Why do we want to use neural networks for MD simulations?

1. Because neural networks remove the need for training data.
2. Neural networks are fast in prediction, differentiable in principle, and can approach QM accuracy when trained on QM data.
3. Force fields do not work in MD simulations.

In [ ]:
answer_md = 0 # please pick your answer
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: answer_md - possible points: 1

# 1 Point
assert answer_md in [1, 2, 3]

##### DO NOT CHANGE #####

We start by loading and inspecting different geometries of methanol. These geometries were sampled by randomly distorting the molecule, producing many conformations with different atomic positions. A common format for molecular coordinates is the `.xyz` file. The format of a single `.xyz` structure is:

1. 1st line: number of atoms (data format: integer)
2. 2nd line: comment (data format: string)
3. 3rd line and following: element symbols and x-, y-, z-coordinates (data format: string and float, respectively)

An example `.xyz` file of methanol and the other data files will be downloaded in the next cell, and we will use them throughout the exercise.

In [ ]:
##### DO NOT CHANGE #####
import hashlib
from pathlib import Path

import requests

molecule_name = "methanol"

_SOURCE_DATA_DIR = Path("assignments/source-ml4nat/ex_06_AtomMD")
DATA_DIR = _SOURCE_DATA_DIR if _SOURCE_DATA_DIR.exists() else Path(".")

DATA_SOURCES = {
    f"{molecule_name}_conformers.xyz": {
        "url": "https://bwsyncandshare.kit.edu/s/cK6H9SHBWse6Tcp/download",
        "size": 1830305,
        "sha256": "270f83ac8eafdd8cbf686de65acbf0b0f4d6de1cca2b7d3f01d26f5232f809d7",
    },
    f"{molecule_name}_coordinates.npy": {
        "url": "https://bwsyncandshare.kit.edu/s/rkaxzcygN9ZzHEk/download",
        "size": 864272,
        "sha256": "240c7592695ede0c9c6fcbf9a935973bc6339940d58c2fd056d76cdcfa980c2b",
    },
    f"{molecule_name}_energy.npy": {
        "url": "https://bwsyncandshare.kit.edu/s/njCdHDy3c2fEmoq/download",
        "size": 48136,
        "sha256": "d427618bb8c4dec0f38d0b81c1f98b2e8bb89376f62d140f669502bc4bdce3fb",
    },
    f"{molecule_name}_gradients.npy": {
        "url": "https://bwsyncandshare.kit.edu/s/WtALXyQCGB8m2Z6/download",
        "size": 864272,
        "sha256": "8bae1842f8bea4158c1b89451ab0265a9df8c63d06fdfd38b0fc5c18d59a50da",
    },
}


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def data_file_is_valid(path: Path, expected_size: int, expected_sha256: str) -> bool:
    if not path.exists():
        return False
    if path.stat().st_size != expected_size:
        print(f"Found {path.name}, but its file size is wrong. Downloading it again.")
        return False
    if file_sha256(path) != expected_sha256:
        print(f"Found {path.name}, but its checksum is wrong. Downloading it again.")
        return False
    return True


def download_data(data_file: str, data_url: str, expected_size: int, expected_sha256: str, timeout: int = 60):
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    path = DATA_DIR / data_file
    if data_file_is_valid(path, expected_size, expected_sha256):
        return

    print(f"Downloading dataset {data_file} from {data_url}...", flush=True, end=" ")
    response = requests.get(data_url, timeout=timeout)
    response.raise_for_status()
    path.write_bytes(response.content)

    if not data_file_is_valid(path, expected_size, expected_sha256):
        raise RuntimeError(
            f"Downloaded {data_file}, but its file size or checksum does not match the expected data."
        )
    print("Done.")


for data_file, metadata in DATA_SOURCES.items():
    download_data(
        data_file,
        metadata["url"],
        metadata["size"],
        metadata["sha256"],
    )

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# Import and install the necessary Python libraries.
%pip install py3Dmol
import numpy as np
from numpy.typing import NDArray
import matplotlib.pyplot as plt

try:
    import py3Dmol
except ModuleNotFoundError:
    py3Dmol = None
    print(
        "py3Dmol is not available in this kernel. The 3D viewer cells will be skipped, "
        "but all graded parts of the exercise can still run."
    )

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# Define a function that reads the xyz file and stores the data in a list.
# We will need this function to read atomic positions and related data files.
def load_csv(filepath, delimiter=" "):
    values = []
    # Open file using a context manager.
    with open(filepath, "r") as file:
        # Read separate entries.
        for line in file:
            line_list = line.strip().split(delimiter)
            line_list = [x.strip() for x in line_list if x != ""]
            values.append(line_list)
    # The file is automatically closed when exiting the 'with' block.
    return values


##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# Here we show the first 30 list entries of the "values" list.
# The 1st entry is the number of atoms in the first molecule, the 2nd one is the comment line,
# and the following entries contain an element symbol plus its coordinates.
lines = load_csv(DATA_DIR / f"{molecule_name}_conformers.xyz")
lines[:30]

##### DO NOT CHANGE #####

In the cell below, assign all molecules to a list called `mols`. Each molecule should be represented as one list with six atoms. Each atom should be represented as one list with four values: the element symbol (`"O"`, `"C"`, or `"H"`) followed by its three Cartesian coordinates. The coordinates should be numeric values; floats are typical, but integer values are also fine when they are numerically correct.

In [ ]:
def lines_to_xyz(values):
    # Parse molecules with flexible size.
    # YOUR CODE HERE
    raise NotImplementedError()
    return convert_list

In [ ]:
##### DO NOT CHANGE #####
mols = lines_to_xyz(lines)

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ID: mols - possible points: 2

# 2 Points
assert (
    np.sum(np.abs(np.array(mols[1][2][1:]) - np.array([-0.2012, 0.0, -1.0612]))) < 0.001
)
assert mols[0][1][0] == "O" and mols[0][3][0] == "H"

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ID: lines_to_mols - possible points: 2

# 2 Points
assert len(mols) == 6001
assert len(mols[0]) == 6

##### DO NOT CHANGE #####

# 1. Learn Energies and Gradients of the Molecule

## 1.1 Load Data

To make the rest of the notebook independent of the first parsing task, we load the prepared NumPy arrays below.

In [ ]:
##### DO NOT CHANGE #####
# Prepared arrays for the rest of the notebook.
import numpy as np

geos = np.load(DATA_DIR / f"{molecule_name}_coordinates.npy")  # in A
energies = np.load(DATA_DIR / f"{molecule_name}_energy.npy")  # in eV
grads = (
    np.load(DATA_DIR / f"{molecule_name}_gradients.npy") * 27.21138624598853 / 0.52917721090380
)  # from H/B to eV/A


elements = []
number_of_elements = geos.shape[1]
with open(DATA_DIR / f"{molecule_name}_conformers.xyz", "r") as xyz_file:
    for lineidx, line in enumerate(xyz_file):
        if lineidx >= 2 and lineidx < number_of_elements + 2:
            elements.append(line.split()[0])
# Look at the shape of the loaded objects.
print("Geometries: ", geos.shape)
print("Energies: ", energies.shape)
print("Gradients: ", grads.shape)
print("Elements: ", elements)

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# Visualize methanol_conformers.xyz using py3Dmol.view.
if py3Dmol is None:
    print("Skipping the 3D viewer because py3Dmol is not available in this kernel.")
else:
    viewer = py3Dmol.view(width=400, height=300)
    viewer.addModelsAsFrames((DATA_DIR / "methanol_conformers.xyz").read_text(), "xyz")
    viewer.setStyle({"stick": {}})
    viewer.zoomTo()
    viewer.animate({"loop": "forward", "reps": 2, "interval": 500})
    viewer.show()

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# Plot the distribution of energies.
plt.figure()
plt.hist(energies)
plt.xlabel("Energy [eV]")
plt.ylabel("Number of occurrences")
plt.show()

##### DO NOT CHANGE #####

The lecture discussed different ways to generate molecular geometries for ML potentials, such as vibration-based sampling, trajectory-based sampling, and large molecular databases. In this notebook, we use sampled conformations of one molecule, methanol.

Which limitation should we keep in mind when training and evaluating a model on this data?

1. The data covers one molecule and only the sampled region of its conformational space.
2. The data automatically generalizes to all organic molecules.
3. The model no longer needs forces because it sees coordinates.
4. The data is independent of the quantum method used to label it.

In [ ]:
answer_sampling_data = 0  # please pick your answer
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: SamplingData - possible points: 1

# 1 Point
assert answer_sampling_data in [1, 2, 3, 4]

##### DO NOT CHANGE #####

## 1.2 Train-Test-Split

In [ ]:
##### DO NOT CHANGE #####
# Libraries for the train-test split, scaling, metrics, and neural-network model.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import tensorflow as tf
import keras

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# Scale energies and make a train-test split.
scaler = StandardScaler(with_std=True, with_mean=True, copy=True)

energy_scaled = scaler.fit_transform(energies[:, np.newaxis])
grads_scaled = grads / scaler.scale_

train_x, test_x, train_g, test_g, train_e, test_e = train_test_split(
    geos, grads_scaled, energy_scaled, test_size=0.2, shuffle=True, random_state=42
)
print(train_x.shape)
print(train_g.shape)
print(train_e.shape)
# Each coordinate value contains 6 atoms with 3 Cartesian coordinates.
# Each gradient value contains dE/dx, dE/dy, and dE/dz for each of the 6 atoms.
# The energy is one global value per molecule.

##### DO NOT CHANGE #####

In principle, you could already set up a TensorFlow-Keras model as shown below, using all Cartesian coordinates directly as input. You can test it with the data above, but please remove your exploratory code again before submission.

Please have a look at the TensorFlow API documentation: https://www.tensorflow.org/api_docs

Here we create a TensorFlow model by sequentially setting up the model layers. The input tensor is passed from layer to layer. The first dimension is always the batch dimension. A fully connected neural network is given by `keras.layers.Dense`. With the input layer we define the input shape from which the hidden weights can be built. Since we want to model a smooth potential, we use `selu` instead of `relu`, and we use a regularizer for the kernel weights.

In [ ]:
##### DO NOT CHANGE #####
model = keras.Sequential()
model.add(keras.Input(shape=(6, 3)))
model.add(keras.layers.Flatten())
model.add(
    keras.layers.Dense(
        300,
        use_bias=True,
        activation="selu",
        kernel_regularizer=keras.regularizers.L1(1e-5),
    )
)
model.add(
    keras.layers.Dense(
        300,
        use_bias=True,
        activation="selu",
        kernel_regularizer=keras.regularizers.L1(1e-5),
    )
)
model.add(keras.layers.Dense(1, use_bias=True, activation="linear"))
model.summary()

##### DO NOT CHANGE #####

What could be the problem with this approach? Choose one of the answers:

1. The model does not have trainable weights.
2. The model cannot deal with input tensors of rank > 1.
3. The model has input features that are not rotation- and translation-invariant, but the model output should be.

In [ ]:
answer_model_1 = 0  # please pick an answer as int
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: answer_model1 - possible points: 1

# 1 Point
assert answer_model_1 in [1, 2, 3]

##### DO NOT CHANGE #####

## 1.3 Features

Deep-learning APIs like TensorFlow can compute gradients and Jacobians analytically. We use this to compute gradients, and therefore forces, from the neural-network potential. This only works cleanly if the feature descriptors are computed inside the model, so TensorFlow can differentiate through them.

Raw Cartesian coordinates are not ideal energy features: translating or rotating the molecule changes the coordinates, but should not change the energy. We will therefore implement an `InverseDistances` layer. Pairwise inverse distances are invariant to translation and rotation.

This is the "easy solution" from the lecture: represent a fixed molecule by a vector of inverse pair distances. It is useful for this exercise, but it is molecule-specific and not transferable to arbitrary molecules. More general learned potentials use local atomic environments, symmetry functions, SchNet-like interaction layers, or graph neural networks.

The backward pass is handled by TensorFlow as long as we use TensorFlow functions with defined gradients: https://www.tensorflow.org/guide/autodiff

In [ ]:
class InverseDistances(keras.layers.Layer):
    def __init__(self, pair_indices=None, **kwargs):
        super().__init__(**kwargs)

        self.pair_indices = pair_indices

    def build(self, input_shape):
        if self.pair_indices is None:
            self.pair_indices = np.array(
                [[i, j] for i in range(input_shape[1]) for j in range(i)],
                dtype=np.int64,
            )
        else:
            self.pair_indices = np.array(self.pair_indices, dtype=np.int64)

        self.tf_pair_indices = self.add_weight(
            name="pair_indices",
            shape=self.pair_indices.shape,
            initializer=keras.initializers.Zeros(),
            dtype="int64",
            trainable=False,
        )

        super().build(input_shape)

        self.set_weights([self.pair_indices])

    def call(self, inputs, **kwargs):
        # Translate the NumPy version of the forward pass to TensorFlow functions.
        # Everything must be vectorized so TensorFlow can differentiate through it.
        # YOUR CODE HERE
        raise NotImplementedError()
        return invd_out

    def _call_numpy_version_not_use(self, inputs):
        # The same computation as call, written in NumPy for visible self-checks.
        cordbatch = inputs
        indexbatch = self.pair_indices
        v1 = np.take(cordbatch, indexbatch[:, 0], axis=1)
        v2 = np.take(cordbatch, indexbatch[:, 1], axis=1)
        vec = v2 - v1
        norm_vec = np.sqrt(np.sum(vec * vec, axis=-1))
        invd_out = 1 / norm_vec
        return invd_out

    def compute_output_shape(self, input_shape: tuple[int, int, int]):
        num_atoms = input_shape[1]
        num_pairs = num_atoms * (num_atoms - 1) // 2
        return (input_shape[0], num_pairs)

In [ ]:
##### DO NOT CHANGE #####
# Test layer
test_layer = InverseDistances(dtype="float64")
test_layer.build(geos[:10, :, :].shape)
result_test_layer = test_layer(geos[:10, :, :]).numpy()
result_test_numpy = test_layer._call_numpy_version_not_use(geos[:10, :, :])
result_test_numpy.shape

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ID: InvdLayer - possible points: 3

# 3 Points
assert np.sum(np.abs(result_test_layer - result_test_numpy)) < 1e-6

##### DO NOT CHANGE #####

The lecture emphasized that inverse-distance features are a useful molecule-specific representation, but not a fully general learned potential. Modern potentials often use local atomic environments, symmetry functions, SchNet-like interaction layers, or graph neural networks so that the model can handle different molecules and larger systems.

Why is the inverse-distance representation in this notebook not easily transferable to ethanol or arbitrary molecules? Choose all correct options:

1. The number of inverse-distance features depends on the number of atoms.
2. The feature vector assumes a fixed atom order.
3. It is not a local atom-centered representation.
4. It is already equivalent to SchNet/message passing.
5. It may extrapolate poorly outside the sampled methanol conformations.

In [ ]:
answer_transferability = []  # choose all correct options as a list of integers
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: Transferability - possible points: 2

# 2 Points
assert isinstance(answer_transferability, (list, tuple, set))
assert len(answer_transferability) > 0

##### DO NOT CHANGE #####

In principle, we can now make a model like the one shown below. However, training with gradient predictions usually leads to better results. Note the non-trainable weights for the pair indices that come from our custom layer.

In [ ]:
##### DO NOT CHANGE #####
model = keras.Sequential()
model.add(keras.Input(shape=(6, 3)))
model.add(InverseDistances(name="InverseDistance"))
model.add(
    keras.layers.Dense(
        300,
        use_bias=True,
        activation="selu",
        kernel_regularizer=keras.regularizers.L1(1e-5),
    )
)
model.add(
    keras.layers.Dense(
        300,
        use_bias=True,
        activation="selu",
        kernel_regularizer=keras.regularizers.L1(1e-5),
    )
)
model.add(keras.layers.Dense(1, use_bias=True, activation="linear"))
model.summary()

##### DO NOT CHANGE #####

Why could it be advantageous to train with energy AND forces as model outputs?

1. The forces are not connected to the energies and therefore need to be in the training set.
2. Adding the forces makes training much faster.
3. The gradients determine the slope of the potential energy surface and act as an additional form of regularization.

In [ ]:
answer_gradient = 0  # select the number of the correct answer
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: AnswerGradient - possible points: 1

# 1 Point
assert answer_gradient in [1, 2, 3]

##### DO NOT CHANGE #####

## 1.4 Energy + Gradients

We can improve gradient prediction by training on energies and gradients simultaneously. To do this, the analytical gradient prediction has to be integrated into the model. Subclassing `keras.Model` allows us to implement a more general model definition than a simple sequential model.

In [ ]:
##### DO NOT CHANGE #####
class EnergyGradientModel(keras.Model):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        self.feat_layer = InverseDistances()

        self.dense_layers = [
            keras.layers.Dense(
                300,
                use_bias=True,
                activation="selu",
                kernel_regularizer=keras.regularizers.L1(1e-5),
            ),
            keras.layers.Dense(
                300,
                use_bias=True,
                activation="selu",
                kernel_regularizer=keras.regularizers.L1(1e-5),
            ),
        ]

        self.energy_layer = keras.layers.Dense(1, use_bias=True, activation="linear")

    def build(self, input_shape):
        self.feat_layer.build(input_shape)
        input_shape = self.feat_layer.compute_output_shape(input_shape)

        for layer in self.dense_layers:
            layer.build(input_shape)
            input_shape = layer.compute_output_shape(input_shape)

        self.energy_layer.build(input_shape)

        super().build(input_shape)

    def call(self, inputs, training=False, **kwargs):
        x = inputs
        with tf.GradientTape() as tape:
            tape.watch(x)
            hidden = self.feat_layer(x)
            for d in self.dense_layers:
                hidden = d(hidden, training=training)
            temp_e = self.energy_layer(hidden)
        temp_g = tape.batch_jacobian(temp_e, x)
        temp_g = temp_g[:, 0, :, :]
        return [temp_e, temp_g]

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
model = EnergyGradientModel()
model.build(train_x.shape)

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# Compile and train the model.
def compile_train_model(
    model: keras.Model,
    x,
    y,
    validation_data=None,
    epochs=1000,
    lr=0.5e-3,
    validation_freq=10,
    batch_size=128,
    verbose=2,
    loss="mean_squared_error",
    metrics=["mean_absolute_error"],
    loss_weights=None,
):
    # Compile model with optimizer and learning rate.
    optimizer = keras.optimizers.Adam(learning_rate=lr)
    model.compile(
        loss=loss,
        optimizer=optimizer,
        metrics=metrics,
        loss_weights=loss_weights,
    )

    hist = model.fit(
        x,
        y,
        epochs=epochs,
        batch_size=batch_size,
        validation_freq=validation_freq,
        validation_data=validation_data,
        verbose=2,
    )

    return hist

##### DO NOT CHANGE #####

Please submit your solution with `do_training = False`.

In [ ]:
do_training = False
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
if do_training:
    hist = compile_train_model(
        model,
        train_x,
        [train_e, train_g],
        validation_data=(test_x, [test_e, test_g]),
        loss=["mean_squared_error", "mean_squared_error"],
        metrics=[["mean_absolute_error"], ["mean_absolute_error"]],
        loss_weights=[1, 5],
    )

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# Some plotting functions.
def plot_history(hist, validation_freq=10, scale=1):
    plt.figure()
    for key, loss in hist.history.items():
        np_loss = np.array(loss)
        if "val" in key:
            plt.plot(
                np.arange(np_loss.shape[0]) * validation_freq + validation_freq,
                np_loss,
                label=key,
            )
        else:
            plt.plot(np.arange(np_loss.shape[0]), np_loss, label=key)

    plt.xlabel("Epochs")
    plt.ylabel("Loss ")
    plt.title("Loss vs. epochs")
    plt.legend(loc="upper right", fontsize="x-small")
    plt.show()


def plot_prediction(pred, true):
    mae_valid = np.mean(np.abs(pred - true))
    r2_data = r2_score(true, pred)
    print("MAE", mae_valid)
    print("r2_score", r2_data)
    plt.figure()
    plt.scatter(
        pred.flatten(),
        true.flatten(),
        alpha=0.3,
        label="MAE: {0:0.4f} \nr2 {1:0.4f}".format(mae_valid, r2_data),
    )
    plt.plot(
        np.linspace(np.amin(true), np.amax(true), 100),
        np.linspace(np.amin(true), np.amax(true), 100),
        color="red",
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.legend(loc="upper left", fontsize="x-large")
    plt.show()

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
true_test = scaler.inverse_transform(test_e)
true_test_grad = test_g * scaler.scale_

if do_training:
    pred_test_scaled, pred_test_grad_scaled = model.predict(test_x)
    pred_test = scaler.inverse_transform(pred_test_scaled)
    pred_test_grad = pred_test_grad_scaled * scaler.scale_

    plot_history(hist, validation_freq=10, scale=scaler.scale_)
    plot_prediction(pred_test, true_test)
    plot_prediction(
        np.reshape(pred_test_grad, (-1, 18)), np.reshape(true_test_grad, (-1, 18))
    )
else:
    # Deterministic reference predictions for grading when training is disabled for submission.
    rng = np.random.default_rng(42)
    pred_test = true_test + 0.02 * np.std(true_test) * rng.standard_normal(true_test.shape)
    pred_test_grad = true_test_grad + 0.02 * np.std(true_test_grad) * rng.standard_normal(
        true_test_grad.shape
    )

##### DO NOT CHANGE #####

Please compute the validation `r2_score` values from the prediction arrays produced above. During exploration you may set `do_training = True` and train your own model. For submission, keep `do_training = False`; the notebook then uses deterministic reference predictions so this scoring task remains reproducible for grading.

Use `r2_score(true_test, pred_test)` for the energy and reshape the gradient arrays to shape `(n_samples, 18)` before computing the gradient score.

In [ ]:
r2_score_energy = 0
r2_score_gradient = 0
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: Testr2score - possible points: 2

# 2 Points
assert np.asarray(r2_score_energy).shape == ()
assert np.asarray(r2_score_gradient).shape == ()
assert np.isfinite(r2_score_energy)
assert np.isfinite(r2_score_gradient)
assert r2_score_energy > 0.90
assert r2_score_gradient > 0.90

##### DO NOT CHANGE #####

# MD Simulation

If we know the forces on each atom, we can update the atoms' velocities and positions. After obtaining the new positions, we calculate the forces again and continue this process. In conventional MD, computing new forces and the corresponding molecular energy can be expensive. With a trained and accurate ML model, we can predict energies and forces much faster.

For time integration, we use [Verlet integration](https://en.wikipedia.org/wiki/Verlet_integration) to integrate Newton's equations of motion.

The standard scheme is:

* Calculate ${\displaystyle {\vec {v}}\left(t+{\tfrac {1}{2}}\,\Delta t\right)={\vec {v}}(t)+{\tfrac {1}{2}}\,{\vec {a}}(t)\,\Delta t}$.

* Calculate ${\displaystyle {\vec {x}}(t+\Delta t)={\vec {x}}(t)+{\vec {v}}\left(t+{\tfrac {1}{2}}\,\Delta t\right)\,\Delta t}$.

* Derive ${\displaystyle {\vec {a}}(t+\Delta t)}$ from the interaction potential using ${\displaystyle {\vec {x}}(t+\Delta t)}$.

* Calculate ${\displaystyle {\vec {v}}(t+\Delta t)={\vec {v}}\left(t+{\tfrac {1}{2}}\,\Delta t\right)+{\tfrac {1}{2}}\,{\vec {a}}(t+\Delta t)\Delta t}$.

Here is a Python implementation with different time-integration schemes.

In [ ]:
##### DO NOT CHANGE #####
# First we need a wrapper around our trained neural network to generate the output needed in MD.
class PotentialNN:
    unit_Bohr_A = 0.52917721090380
    unit_Hartree_eV = 27.21138624598853

    def __init__(self, model, scaler):
        self.model = model
        self.scaler = scaler

    def __call__(self, x):
        # Convert coordinates from Bohr to Angstrom.
        x_A = x * self.unit_Bohr_A
        # Call model.
        eng, grad = self.model.predict(x_A)
        # Undo the scaling.
        eng = self.scaler.inverse_transform(eng)
        grad = grad * self.scaler.scale_
        # Convert units to Hartree and Hartree/Bohr.
        eng_B = eng / self.unit_Hartree_eV
        grad_BH = grad / self.unit_Hartree_eV * self.unit_Bohr_A
        return eng_B, grad_BH

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# Then we need an MD ensemble.


class BatchEnsemble:
    unit_Bohr_A = 0.52917721090380
    unit_Bohr_m = 5.2917721090380e-11
    unit_me_kg = 9.109383701528e-31
    unit_Hartree_eV = 27.21138624598853
    unit_Hartree_J = 4.359744722207185e-18
    unit_atu_s = 2.418884326585747e-17
    unit_1u_me = 1.6605390666050e-27 / 9.109383701528e-31

    const_kB = 8.617333262e-5 / 27.21138624598853
    sign_force = -1.0

    def __init__(self, coord, mass, potential, velo=None):
        """Initial settings. All properties are in atomic units.

        Args:
            coord (np.array): Initial position of shape (batch, N, 3) in [Bohr]
            mass (np.array): Mass of particles (batch, N, 1) in [me]
            potential (callable): Potential model inputs coord and returns energies and gradients
                                  of shape (batch, 1) and (batch, N, 3) in [Hartree] and [Hartree/Bohr]
            velo (np.array): Initial velocities of shape (batch, N, 3)
        """

        # System properties
        self.mass = mass
        self.potential = potential
        self.number_particles = self.mass.shape[1]

        # Trajectory properties
        self.traj_x = []  # position in Bohr
        self.traj_v = []  # velocity in Bohr/atu
        self.traj_a = []  # acceleration in Bohr/atu^2
        self.traj_t = []  # time in atu
        self.traj_p = []  # momentum in Bohr*me/atu
        self.traj_F = []  # force in Hartree/Bohr
        self.traj_E = []  # potential energy in Hartree
        self.traj_E_kin = []  # kinetic energy in Hartree
        self.traj_T = []  # temperature in K

        #######################################################################

        # Set initial values i.e. traj[0]
        initial_x = coord
        if velo is None:
            initial_v = np.zeros_like(coord)
        else:
            initial_v = velo
        initial_p = initial_v * self.mass
        initial_eng, initial_grad = self.potential(initial_x)
        initial_force = self.sign_force * initial_grad
        initial_eng_kin = np.sum(
            np.sum(0.5 * self.mass * initial_v * initial_v, axis=-1),
            axis=-1,
            keepdims=True,
        )
        initial_temp = 2 / 3 * initial_eng_kin / self.number_particles / self.const_kB
        initial_a = initial_force / self.mass

        # Add initial values to trajectory
        self.add_point(
            t=0.0,
            x=initial_x,
            v=initial_v,
            a=initial_a,
            F=initial_force,
            E=initial_eng,
            E_kin=initial_eng_kin,
            T=initial_temp,
            p=initial_p,
        )

    def add_point(
        self,
        t: float,
        x: NDArray,
        v: NDArray,
        a: NDArray,
        F: NDArray,
        E: NDArray,
        E_kin: NDArray,
        T: NDArray,
        p: NDArray,
    ):
        """Add point to trajectory.

        Args:
            t (float): Time in atu
            x (np.array): Position in Bohr
            v (np.array): Velocity in Bohr/atu
            a (np.array): Acceleration in Bohr/atu^2
            F (np.array): Force in Hartree/Bohr
            E (np.array): Potential energy in Hartree
            E_kin (np.array): Kinetic energy in Hartree
            T (np.array): Temperature in K
            p (np.array): Momentum in Bohr*me/atu
        """
        self.traj_t.append(t)
        self.traj_x.append(x)
        self.traj_v.append(v)
        self.traj_a.append(a)
        self.traj_F.append(F)
        self.traj_E.append(E)
        self.traj_E_kin.append(E_kin)
        self.traj_T.append(T)
        self.traj_p.append(p)

    def initialize_velocity(self, T):
        """Overwrite initial velocity by Boltzmann distribution.

        Args:
            T (float): Temperature in K
        """
        initial_velo = np.random.standard_normal(self.traj_x[0].shape)
        initial_velo = initial_velo * np.sqrt(self.const_kB * T / self.mass)
        initial_eng_kin = np.sum(
            np.sum(0.5 * self.mass * initial_velo * initial_velo, axis=-1),
            axis=-1,
            keepdims=True,
        )
        initial_p = initial_velo * self.mass
        initial_T = 2 / 3 * initial_eng_kin / self.number_particles / self.const_kB

        self.traj_v[0] = initial_velo
        self.traj_E_kin[0] = initial_eng_kin
        self.traj_T[0] = initial_T
        self.traj_p[0] = initial_p

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# This class is an MD propagator that repeatedly advances the simulation by time steps.
class TimeIntegration:
    unit_atu_s = 2.418884326585747e-17
    const_kB = 8.617333262e-5 / 27.21138624598853

    def __init__(self, ensemble: BatchEnsemble):
        self.ensemble = ensemble

    def propagate_timestep(self):
        raise NotImplementedError("This method should be implemented in a subclass")

    def propagate(self, time_length, step_size):
        """Propagate ensemble.

        Args:
            time_length: Time of the simulation in [fs]
            step_size: Time step in [fs]
        """
        # Repeat time step.
        num_steps = int(time_length / step_size)

        # Map to atomic time units.
        time_length = time_length * 1e-15 / self.unit_atu_s
        step_size = step_size * 1e-15 / self.unit_atu_s
        print("Run MD for:", time_length, "a.t.u with steps:", step_size, "a.t.u")

        # Run MD.
        for i in range(num_steps):
            self.propagate_timestep(step_size)
            if i % 500 == 0:
                print("Steps done:", i)

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# One parameter we have to consider is how to control the temperature.
# The next two classes show two options, and the visualizations reveal the difference.
class VerletIntegration(TimeIntegration):
    def __init__(self, ensemble: BatchEnsemble):
        super().__init__(ensemble)

    def propagate_timestep(self, delta_t):
        # time in atu

        t = self.ensemble.traj_t[-1]
        x_t = self.ensemble.traj_x[-1]
        a_t = self.ensemble.traj_a[-1]
        v_t = self.ensemble.traj_v[-1]
        m = self.ensemble.mass
        N = self.ensemble.number_particles
        kB = self.const_kB
        sig_F = self.ensemble.sign_force

        v_t_dt_2 = v_t + 0.5 * a_t * delta_t
        x_t_dt = x_t + v_t_dt_2 * delta_t

        e_t_dt, g_t_dt = self.ensemble.potential(x_t_dt)
        a_t_dt = sig_F * g_t_dt / m

        v_t_dt = v_t_dt_2 + 0.5 * a_t_dt * delta_t
        e_kin_t_dt = np.sum(
            np.sum(0.5 * m * v_t_dt * v_t_dt, axis=-1), axis=-1, keepdims=True
        )
        p_t_dt = v_t_dt * m
        T_dt = 2 / 3 * e_kin_t_dt / N / kB

        self.ensemble.add_point(
            t=t + delta_t,
            x=x_t_dt,
            v=v_t_dt,
            a=a_t_dt,
            F=sig_F * g_t_dt,
            E=e_t_dt,
            E_kin=e_kin_t_dt,
            T=T_dt,
            p=p_t_dt,
        )

##### DO NOT CHANGE #####

A thermostat allows us to control the temperature of the system during the simulation.

In [ ]:
##### DO NOT CHANGE #####
class BerendsenThermostat(TimeIntegration):
    def __init__(self, ensemble: BatchEnsemble, f_cool=0, T0=300):
        """Make time integration with a Berendsen thermostat for Verlet dynamics.

        Args:
            ensemble: BatchEnsemble class
            f_cool: Cooling coupling as frequency in [fs]
            T0: Temperature of the bath [K]
        """
        self.ensemble = ensemble
        self.f_cool = f_cool / 1e-15 * self.unit_atu_s
        self.T0 = T0

    def propagate_timestep(self, delta_t):
        # time in atu

        t = self.ensemble.traj_t[-1]
        x_t = self.ensemble.traj_x[-1]
        a_t = self.ensemble.traj_a[-1]
        v_t = self.ensemble.traj_v[-1]
        m = self.ensemble.mass
        N = self.ensemble.number_particles
        kB = self.const_kB
        sig_F = self.ensemble.sign_force
        f_cool = self.f_cool

        v_t_dt_2 = v_t + 0.5 * a_t * delta_t

        e_kin_t_dt_2 = np.sum(
            np.sum(0.5 * m * v_t_dt_2 * v_t_dt_2, axis=-1), axis=-1, keepdims=True
        )
        T_dt_2 = 2 / 3 * e_kin_t_dt_2 / N / kB
        lamd = np.sqrt(1 + f_cool * delta_t * (self.T0 / T_dt_2 - 1))

        x_t_dt = x_t + v_t_dt_2 * delta_t

        e_t_dt, g_t_dt = self.ensemble.potential(x_t_dt)
        a_t_dt = sig_F * g_t_dt / m

        v_t_dt = v_t_dt_2 + 0.5 * a_t_dt * delta_t
        v_t_dt = np.expand_dims(lamd, axis=-1) * v_t_dt
        e_kin_t_dt = np.sum(
            np.sum(0.5 * m * v_t_dt * v_t_dt, axis=-1), axis=-1, keepdims=True
        )
        p_t_dt = v_t_dt * m
        T_dt = 2 / 3 * e_kin_t_dt / N / kB

        # Add time step.
        self.ensemble.add_point(
            t=t + delta_t,
            x=x_t_dt,
            v=v_t_dt,
            a=a_t_dt,
            F=sig_F * g_t_dt,
            E=e_t_dt,
            E_kin=e_kin_t_dt,
            T=T_dt,
            p=p_t_dt,
        )

##### DO NOT CHANGE #####

Now we can run a very simple MD simulation for a batch of molecules. We take the energy minimum and initialize a batch of molecules at temperature $T$. We also have to provide the atom masses and the neural-network potential in matching units. Please submit your notebook with `do_propagate = False` and `show_trajectory = False`.

In [ ]:
do_propagate = False
show_trajectory = False
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# Make a batch of molecules.
lowest = np.argsort(energies)[0]
print("Energy Minimum", energies[lowest])
x_md = geos[lowest]
x_md = np.repeat(np.expand_dims(x_md, axis=0), 10, axis=0)
x_md = x_md / PotentialNN.unit_Bohr_A
mass = (
    np.array([[[12.0], [15.99491], [1.007825], [1.007825], [1.007825], [1.007825]]])
    * BatchEnsemble.unit_1u_me
)

# Set up ensemble and MD simulation.
potential = PotentialNN(model, scaler)
ensemble_md = BatchEnsemble(x_md, mass, potential)
ensemble_md.initialize_velocity(1000.0)
trajectory_md = VerletIntegration(ensemble_md)

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
if do_propagate:
    trajectory_md.propagate(1000, 1)
    plt.figure(figsize=(15, 5))
    plt.subplot(121)
    for i in range(10):
        plt.plot(
            np.array(ensemble_md.traj_t),
            np.array(ensemble_md.traj_E)[:, i, 0],
            label=str(i),
        )
    plt.legend(loc="right")
    plt.xlabel("Time [a.t.u]")
    plt.ylabel("Potential Energy [Hartree]")
    plt.subplot(122)
    for i in range(10):
        plt.plot(
            np.array(ensemble_md.traj_t),
            np.array(ensemble_md.traj_T)[:, i, 0],
            label=str(i),
        )
    plt.legend(loc="right")
    plt.xlabel("Time [a.t.u]")
    plt.ylabel("Temperature [K]")
    plt.ylim([0, 2000])
    plt.show()

##### DO NOT CHANGE #####

Make sure you inspect the graph at the end of the previous cell when you run the propagation during exploration. You can also use the py3Dmol viewer to look at one molecule from the batch; this works in JupyterLab as long as py3Dmol is installed and imported.

In [ ]:
##### DO NOT CHANGE #####
# Export a series of molecules to an xyz file.
def exportXYZs(coords_all, elements_all, filename):
    outfile = open(filename, "w")
    for molidx in range(len(coords_all)):
        outfile.write("%i\n\n" % (len(elements_all[molidx])))
        for atomidx, atom in enumerate(coords_all[molidx]):
            outfile.write(
                "%s %f %f %f\n"
                % (
                    elements_all[molidx][atomidx].capitalize(),
                    atom[0],
                    atom[1],
                    atom[2],
                )
            )
    outfile.close()


if show_trajectory:
    elements_traj = [elements for i in ensemble_md.traj_x]
    coords_traj = [coords[0] * PotentialNN.unit_Bohr_A for coords in ensemble_md.traj_x]
    exportXYZs(coords_traj, elements_traj, "MD_traj_1.xyz")

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
if show_trajectory and py3Dmol is None:
    print("Skipping the trajectory viewer because py3Dmol is not available in this kernel.")
elif show_trajectory:
    viewer = py3Dmol.view(width=600, height=300)
    viewer.addModelsAsFrames(open("MD_traj_1.xyz", "r").read(), "xyz")
    viewer.setStyle({"stick": {}})
    viewer.zoomTo()
    viewer.animate({"loop": "forward", "reps": 1, "interval": 100})
    viewer.show()

##### DO NOT CHANGE #####

You may notice a large spread in the initial temperatures. Look at the method `initialize_velocity` in the definition of the `BatchEnsemble` class. What could be a better way to initialize the velocities to achieve a specific kinetic energy or temperature?

1. Set all $\forall v \in \mathbb{R}^3: v_{x,y,z} = v_0 $ constant.
2. Choose a $|v|$ from the Maxwell-Boltzmann distribution and a random direction for each atom, remove the center-of-mass velocity, and rescale according to the desired temperature.
3. Set all $v = 0$ and adjust the potential scale.

In [ ]:
answer_md_v = 0  # choose the correct answer
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: answermd - possible points: 1

# 1 Point
assert answer_md_v in [1, 2, 3]

##### DO NOT CHANGE #####

Now we run the MD simulation with a thermostat. This means we can cool the system during the simulation.

In [ ]:
##### DO NOT CHANGE #####
potential = PotentialNN(model, scaler)
ensemble_md = BatchEnsemble(x_md, mass, potential)
ensemble_md.initialize_velocity(1000.0)
trajectory_md = BerendsenThermostat(ensemble_md, f_cool=0.01, T0=5)

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
if do_propagate:
    trajectory_md.propagate(2000, 1)
    plt.figure(figsize=(15, 5))
    plt.subplot(121)
    for i in range(10):
        plt.plot(
            np.array(ensemble_md.traj_t),
            np.array(ensemble_md.traj_E)[:, i, 0],
            label=str(i),
        )
    plt.legend(loc="right")
    plt.xlabel("Time [a.t.u]")
    plt.ylabel("Potential Energy [Hartree]")
    plt.subplot(122)
    for i in range(10):
        plt.plot(
            np.array(ensemble_md.traj_t),
            np.array(ensemble_md.traj_T)[:, i, 0],
            label=str(i),
        )
    plt.legend(loc="right")
    plt.xlabel("Time [a.t.u]")
    plt.ylabel("Temperature [K]")
    plt.ylim([0, 2000])
    plt.show()

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
if show_trajectory:
    elements_traj = [elements for i in ensemble_md.traj_x]
    coords_traj = [coords[0] * PotentialNN.unit_Bohr_A for coords in ensemble_md.traj_x]
    exportXYZs(coords_traj, elements_traj, "MD_traj_2.xyz")

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# Like before, make sure you have taken a look at the graph after propagation.
if show_trajectory and py3Dmol is None:
    print("Skipping the trajectory viewer because py3Dmol is not available in this kernel.")
elif show_trajectory:
    viewer = py3Dmol.view(width=600, height=300)
    viewer.addModelsAsFrames(open("MD_traj_2.xyz", "r").read(), "xyz")
    viewer.setStyle({"stick": {}})
    viewer.zoomTo()
    viewer.animate({"loop": "forward", "reps": 1, "interval": 25})
    viewer.show()

##### DO NOT CHANGE #####

Compare the time evolution of the simulation with Verlet integration to the one with the Berendsen thermostat. Which one shows a decrease in the temperature and energy of the system? Answer with the string `"A"` for Verlet or `"B"` for Berendsen. Hint: you can also check the visualization and observe the molecule's vibration over time.

In [ ]:
answer_observation = ""
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: Answer_Thermo - possible points: 1

# 1 Point
assert isinstance(answer_observation, str)

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ID: AllOff - possible points: 1

# 1 Point
assert do_training == False
assert do_propagate == False
assert show_trajectory == False

##### DO NOT CHANGE #####